# Stock-Prob — Auth Wizard
Jalankan **berurutan** cell di bawah. Tiap cell akan memunculkan popup OAuth di browser Colab.

Urutan: **1 Drive → 2 Google user (Sheets) → 3 GitHub device OAuth → 4 verifikasi**

In [ ]:
# === 1/4 GOOGLE DRIVE ===
print('Popup Drive akan muncul — pilih akun Google kamu lalu Allow')
from google.colab import drive
drive.mount('/content/drive')
import os
print('OK Drive:', os.path.exists('/content/drive/MyDrive'))
print(os.listdir('/content/drive/MyDrive')[:15] if os.path.exists('/content/drive/MyDrive') else 'FAILED')

In [ ]:
# === 2/4 GOOGLE USER AUTH (Sheets / gspread / ADC) ===
print('Popup Google login akan muncul — Allow')
from google.colab import auth
auth.authenticate_user()
import google.auth
creds, project = google.auth.default()
print('OK creds type:', type(creds).__name__)
print('project:', project)

# Test gspread
import gspread
gc = gspread.authorize(creds)
print('OK gspread authorized')
# create or open project sheet
try:
    sh = gc.open('stock-prob-ledger')
    print('Opened existing sheet:', sh.url)
except gspread.SpreadsheetNotFound:
    sh = gc.create('stock-prob-ledger')
    print('Created sheet:', sh.url)
print('Sheet URL:', sh.url)

In [ ]:
# === 3/4 GITHUB OAUTH (device flow, no PAT) ===
import subprocess, re, time, sys
print('Memulai GitHub device OAuth...')
print('Akan muncul URL + code — buka di browser, login GitHub, masukkan code.')

# Non-interactive-ish: use gh auth login with web/device
proc = subprocess.Popen(
    ['gh', 'auth', 'login', '-h', 'github.com', '-p', 'https', '-w', '--skip-ssh-key', '--insecure-storage'],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
# gh -w may still need input; alternative device flow:
proc.kill()

proc = subprocess.Popen(
    ['gh', 'auth', 'login', '-h', 'github.com', '-p', 'https', '--skip-ssh-key', '--insecure-storage'],
    stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
# Send answers: GitHub.com already set via -h, HTTPS via -p, authenticate Git? Y, Login with browser? might prompt
# For newer gh, device flow: paste prompts
out_lines = []
import select
deadline = time.time() + 5
time.sleep(1)
# Try feeding defaults for interactive prompts
try:
    proc.stdin.write('Y\n')
    proc.stdin.flush()
except Exception:
    pass

print('Jika command di bawah tidak memunculkan code, jalankan di cell berikutnya yang manual.')
print('---')
!gh auth login -h github.com -p https -w --skip-ssh-key --insecure-storage

In [ ]:
# === 3b/4 GITHUB (manual device — jalankan jika cell 3 gagal) ===
# Di terminal Colab / cell ini, ikuti URL + one-time code.
!gh auth login --hostname github.com --git-protocol https --web --skip-ssh-key --insecure-storage
!gh auth status

In [ ]:
# === 4/4 VERIFY + SYNC PROJECT ===
import os, shutil
from pathlib import Path

checks = {
    'Drive mounted': os.path.exists('/content/drive/MyDrive'),
    'Local project': os.path.exists('/content/stock-prob'),
}

# Google user creds
try:
    import google.auth
    c, p = google.auth.default()
    checks['Google ADC user-ish'] = 'compute' not in type(c).__name__.lower() or os.path.exists('/content/drive/MyDrive')
    checks['creds_type'] = type(c).__name__
except Exception as e:
    checks['Google ADC'] = str(e)

# GitHub
import subprocess
r = subprocess.run(['gh', 'auth', 'status'], capture_output=True, text=True)
checks['GitHub'] = r.returncode == 0
print(r.stdout or r.stderr)

# Sync project to Drive if mounted
if checks['Drive mounted']:
    dst = Path('/content/drive/MyDrive/stock-prob')
    src = Path('/content/stock-prob')
    dst.mkdir(parents=True, exist_ok=True)
    for sub in ['data', 'predictions', 'exports', 'src', 'notebooks', 'docs']:
        (dst/sub).mkdir(parents=True, exist_ok=True)
    # copy files if not exists or update
    if src.exists():
        for p in src.rglob('*'):
            if p.is_file():
                rel = p.relative_to(src)
                target = dst / rel
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, target)
        print('Synced /content/stock-prob →', dst)
    checks['Drive project'] = dst.exists()

print('=== SUMMARY ===')
for k,v in checks.items():
    print(f'  {k}: {v}')